# Pipeline do SignClassifier


## Imports e Dataset

In [12]:
from pathlib import Path
import pandas as pd
from preprocessor.landmark_processor import LandmarkProcessor

RAIZ = next(
    pasta for pasta in [Path.cwd(), *Path.cwd().parents]
    if (pasta / "pyproject.toml").is_file() and (pasta / "src").is_dir()
)
DATASET = RAIZ / "data" / "minds_libras"

print("Dataset:", DATASET)


Dataset: /home/mhab/VSCode/Libras-Translator/data/minds_libras


## Exemplo de Vídeo

In [10]:
manifesto = pd.read_csv(DATASET / "metadata" / "manifest.csv", dtype=str)

SAMPLE_ID = "01AcontecerSinalizador01-1"
amostra = manifesto.set_index("sample_id").loc[SAMPLE_ID]

VIDEO = DATASET / amostra["path"]
CSV_LANDMARKS = DATASET / "processed" / "landmarks" / f"{SAMPLE_ID}.csv"

amostra.to_frame(name="valor")


,valor
class_id,01
label,Acontecer
signer_id,01
repetition,1
path,raw/01AcontecerSinalizador01-1.mp4
set,train


## Extração de Landmarks

In [11]:
landmarks = pd.read_csv(CSV_LANDMARKS)

print("Amostra:", SAMPLE_ID)
print("Sinal:", amostra["label"])
print("Frames:", landmarks.shape[0])
print("Colunas:", landmarks.shape[1])

landmarks.head(30).round(4)


Amostra: 01AcontecerSinalizador01-1
Sinal: Acontecer
Frames: 104
Colunas: 1087


,frame,pose_0_x,pose_0_y,pose_1_x,pose_1_y,pose_2_x,pose_2_y,pose_3_x,pose_3_y,pose_4_x,...,right_hand_16_x,right_hand_16_y,right_hand_17_x,right_hand_17_y,right_hand_18_x,right_hand_18_y,right_hand_19_x,right_hand_19_y,right_hand_20_x,right_hand_20_y
0,0,0.5249,0.3772,0.5374,0.3419,0.5435,0.3422,0.5500,0.3429,0.5115,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0.5258,0.3795,0.5375,0.3449,0.5435,0.3452,0.5501,0.3458,0.5123,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,0.5260,0.3798,0.5375,0.3456,0.5435,0.3459,0.5501,0.3465,0.5124,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,0.5260,0.3797,0.5375,0.3457,0.5435,0.3461,0.5501,0.3467,0.5124,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,0.5260,0.3795,0.5375,0.3457,0.5435,0.3461,0.5502,0.3467,0.5125,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,0.5260,0.3794,0.5375,0.3457,0.5435,0.3461,0.5502,0.3467,0.5125,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,0.5260,0.3792,0.5375,0.3456,0.5435,0.3461,0.5503,0.3467,0.5125,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,0.5260,0.3791,0.5375,0.3457,0.5435,0.3461,0.5503,0.3468,0.5126,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,8,0.5259,0.3791,0.5374,0.3458,0.5435,0.3463,0.5503,0.3471,0.5126,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,0.5259,0.3791,0.5374,0.3461,0.5435,0.3468,0.5503,0.3477,0.5126,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Seleção de Pontos importantes ASL-2nd

In [13]:
processor = LandmarkProcessor(
    dataset_root=DATASET,
    subset="asl_2nd",
)

# Selecionar os pontos, preservando as coordenadas ausentes.
pontos_asl = processor.select(SAMPLE_ID)

# Recuperar os nomes das coordenadas na ordem usada pelo processor.
colunas_asl = [
    f"{grupo}_{ponto}_{eixo}"
    for grupo, pontos in processor.landmark_groups
    for ponto in pontos
    for eixo in ("x", "y")
]

# Transformar (frames, pontos, coordenadas) em uma tabela.
tabela_asl = pd.DataFrame(
    pontos_asl.reshape(len(pontos_asl), -1),
    columns=colunas_asl,
    index=landmarks["frame"],
)
tabela_asl.index.name = "frame"

print("Formato (frames, pontos, coordenadas):", pontos_asl.shape)
print("Dimensões da tabela:", tabela_asl.shape)

tabela_asl.head(30).round(4)

Formato (frames, pontos, coordenadas): (104, 80, 2)
Dimensões da tabela: (104, 160)


,face_40_x,face_40_y,face_37_x,face_37_y,face_267_x,face_267_y,face_270_x,face_270_y,face_291_x,face_291_y,...,right_hand_16_x,right_hand_16_y,right_hand_17_x,right_hand_17_y,right_hand_18_x,right_hand_18_y,right_hand_19_x,right_hand_19_y,right_hand_20_x,right_hand_20_y
frame,,,,,,,,,,,,,,,,,,,,,
0,0.5102,0.4084,0.5190,0.4030,0.5284,0.4032,0.5360,0.4091,0.5387,0.4136,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.5119,0.4089,0.5203,0.4032,0.5294,0.4030,0.5368,0.4083,0.5395,0.4123,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.5117,0.4077,0.5201,0.4018,0.5291,0.4017,0.5365,0.4075,0.5391,0.4118,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.5114,0.4078,0.5198,0.4019,0.5290,0.4019,0.5364,0.4077,0.5390,0.4122,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.5113,0.4073,0.5197,0.4010,0.5291,0.4008,0.5366,0.4068,0.5393,0.4116,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0.5116,0.4071,0.5201,0.4007,0.5294,0.4005,0.5371,0.4065,0.5398,0.4111,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.5115,0.4065,0.5200,0.4000,0.5293,0.3997,0.5370,0.4056,0.5397,0.4103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.5115,0.4053,0.5200,0.3990,0.5294,0.3987,0.5370,0.4044,0.5397,0.4089,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.5113,0.4040,0.5198,0.3976,0.5292,0.3974,0.5369,0.4033,0.5395,0.4080,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Interpolação